# 📊 DASHBOARD VISUALISASI SKRIPSI — STYLEGAN2-ADA (KAGGLE)
> 10 Visualisasi Analitik + 2 Bonus Aset. Tambahkan dataset hasil training Anda via **Add Data** sebelum Run All.

### ⚙️ 0. Setup Library & Load Data

In [ ]:
import os, glob, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from PIL import Image
warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

# Auto-cari folder hasil training di /kaggle/input
log_files = glob.glob('/kaggle/input/**/log.txt', recursive=True)
fid_files = glob.glob('/kaggle/input/**/metric-fid50k_full.jsonl', recursive=True)
fake_imgs = sorted(glob.glob('/kaggle/input/**/fakes*.png', recursive=True))

print(f'Log ditemukan: {len(log_files)}')
print(f'FID ditemukan: {len(fid_files)}')
print(f'Gambar fakes ditemukan: {len(fake_imgs)}')

In [ ]:
# Harvest semua metrik dari log.txt
ticks, loss_fake, loss_real, sec_kimg, aug_p = [], [], [], [], []
signs_real, signs_fake, r1_pen, pl_pen = [], [], [], []
cpu_mem, gpu_mem = [], []

for lf in sorted(log_files):
    with open(lf) as f:
        for line in f:
            if not line.startswith('{"tick"'): continue
            d = json.loads(line)
            ticks.append(d.get('kimg', 0))
            loss_fake.append(d.get('Loss/scores/fake', 0))
            loss_real.append(d.get('Loss/scores/real', 0))
            sec_kimg.append(d.get('Timing/sec_per_kimg', d.get('sec/kimg', 0)))
            aug_p.append(d.get('Progress/augment', d.get('Augment/p', 0)))
            signs_real.append(d.get('Loss/signs/real', 0))
            signs_fake.append(d.get('Loss/signs/fake', 0))
            r1_pen.append(d.get('Loss/r1_penalty', 0))
            pl_pen.append(d.get('Loss/pl_penalty', 0))
            cpu_mem.append(d.get('Resources/cpu_mem_gb', 0))
            gpu_mem.append(d.get('Resources/peak_gpu_mem_gb', 0))

df = pd.DataFrame({'Kimg':ticks,'Loss_Fake':loss_fake,'Loss_Real':loss_real,
    'Sec_Per_Kimg':sec_kimg,'Augment_P':aug_p,'Sign_Real':signs_real,
    'Sign_Fake':signs_fake,'R1':r1_pen,'PL':pl_pen,'CPU':cpu_mem,'GPU':gpu_mem})

# Harvest FID
fid_kimg, fid_scores = [], []
for ff in sorted(fid_files):
    with open(ff) as f:
        for line in f:
            d = json.loads(line)
            if 'results' in d and 'fid50k_full' in d['results']:
                pkl = d.get('snapshot_pkl', '')
                try: k = int(pkl.split('-')[-1].split('.')[0])
                except: k = len(fid_kimg) * 25
                fid_kimg.append(k)
                fid_scores.append(d['results']['fid50k_full'])

df_fid = pd.DataFrame({'Kimg': fid_kimg, 'FID': fid_scores})
print(f'Total log entries: {len(df)} | FID entries: {len(df_fid)}')
if len(df_fid): print(f'Best FID: {df_fid.FID.min():.2f}')

### 📈 A. Dinamika Pelatihan

In [ ]:
# VIS 1: Loss Discriminator vs Generator
if len(df):
    plt.figure(figsize=(14, 5))
    sns.lineplot(x='Kimg', y='Loss_Real', data=df, label='Discriminator (Real)', color='#1f77b4', linewidth=2, alpha=0.8)
    sns.lineplot(x='Kimg', y='Loss_Fake', data=df, label='Generator (Fake)', color='#ff7f0e', linewidth=2, alpha=0.8)
    plt.title('Vis 1: Kurva Loss — Discriminator vs Generator (Nash Equilibrium)')
    plt.xlabel('Kimg'); plt.ylabel('Loss Score')
    plt.legend(); plt.tight_layout(); plt.show()
else:
    print('Tambahkan dataset log training Anda via Add Data!')

In [ ]:
# VIS 2: FID Score Progression
if len(df_fid):
    plt.figure(figsize=(14, 5))
    sns.lineplot(x='Kimg', y='FID', data=df_fid, marker='o', color='#2ca02c', linewidth=2.5, markersize=8)
    plt.axhline(50, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Target FID < 50')
    best = df_fid.FID.idxmin()
    plt.annotate(f"Terbaik: {df_fid.FID.min():.2f}",
        xy=(df_fid.iloc[best].Kimg, df_fid.iloc[best].FID),
        xytext=(df_fid.iloc[best].Kimg, df_fid.iloc[best].FID + 8),
        arrowprops=dict(facecolor='black', shrink=0.05), fontsize=12, fontweight='bold', ha='center')
    plt.title('Vis 2: Progres Skor FID (Semakin Rendah = Semakin Baik)')
    plt.xlabel('Kimg'); plt.ylabel('FID Score')
    plt.legend(); plt.tight_layout(); plt.show()
else:
    print('Data FID tidak ditemukan')

In [ ]:
# VIS 3 & 4: ADA Augmentation & Computational Efficiency
if len(df):
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    sns.lineplot(x='Kimg', y='Augment_P', data=df, ax=axes[0], color='#9467bd', linewidth=2)
    axes[0].fill_between(df.Kimg, df.Augment_P, alpha=0.2, color='#9467bd')
    axes[0].set_title('Vis 3: Intensitas Augmentasi ADA (p-value)')
    axes[0].set_xlabel('Kimg'); axes[0].set_ylabel('Probabilitas Augmentasi')
    sns.lineplot(x='Kimg', y='Sec_Per_Kimg', data=df, ax=axes[1], color='#8c564b', linewidth=2)
    axes[1].axhline(df.Sec_Per_Kimg.mean(), color='black', linestyle=':', label=f'Rata-rata: {df.Sec_Per_Kimg.mean():.1f}s')
    axes[1].set_title('Vis 4: Efisiensi Komputasi (Detik/Kimg)')
    axes[1].set_xlabel('Kimg'); axes[1].set_ylabel('Detik per Kimg')
    axes[1].legend()
    plt.tight_layout(); plt.show()

### 🖼️ B. Kualitas Visual

In [ ]:
# VIS 5: Komparasi Kualitas Visual Awal vs Akhir
if len(fake_imgs) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(18, 9))
    axes[0].imshow(mpimg.imread(fake_imgs[0]))
    axes[0].set_title('Kimg Awal — Noise Acak', fontsize=16); axes[0].axis('off')
    axes[1].imshow(mpimg.imread(fake_imgs[-1]))
    axes[1].set_title('Kimg Akhir — Struktur Medis Jelas', fontsize=16, fontweight='bold'); axes[1].axis('off')
    plt.suptitle('Vis 5: Komparasi Kualitas Generator (Awal vs Akhir)', fontsize=18, fontweight='bold')
    plt.tight_layout(); plt.show()
elif len(fake_imgs) == 1:
    plt.figure(figsize=(10,10))
    plt.imshow(mpimg.imread(fake_imgs[0])); plt.axis('off')
    plt.title('Vis 5: Hasil Generator Terbaru'); plt.tight_layout(); plt.show()
else:
    print('Gambar fakes*.png tidak ditemukan di dataset input')

In [ ]:
# VIS 6: Distribusi Warna RGB Generator
if len(fake_imgs):
    img_arr = np.array(Image.open(fake_imgs[-1]).convert('RGB'))
    pixels = img_arr.reshape(-1, 3)
    sample = pixels[np.random.choice(len(pixels), size=min(10000, len(pixels)), replace=False)]
    plt.figure(figsize=(12, 5))
    for i, (col, label) in enumerate(zip(['red','green','blue'], ['R','G','B'])):
        sns.kdeplot(sample[:, i], label=f'Channel {label}', color=col, fill=True, alpha=0.3)
    plt.title('Vis 6: Distribusi Intensitas RGB Gambar Generator (Fake)')
    plt.xlabel('Intensitas Piksel (0-255)'); plt.ylabel('Kepadatan')
    plt.legend(); plt.tight_layout(); plt.show()

### 🧬 C. Analisis Tingkat Lanjut (Deep Analysis)

In [ ]:
# VIS 7: Discriminator Sign Accuracy
if len(df) and df.Sign_Real.abs().sum() > 0:
    plt.figure(figsize=(14, 5))
    sns.lineplot(x='Kimg', y='Sign_Real', data=df, label='Real Sign', color='green', linewidth=2, alpha=0.8)
    sns.lineplot(x='Kimg', y='Sign_Fake', data=df, label='Fake Sign', color='red', linewidth=2, alpha=0.8)
    plt.axhline(0, color='black', linestyle='--', alpha=0.4)
    plt.title('Vis 7: Akurasi Deteksi Diskriminator (Real vs Fake)')
    plt.xlabel('Kimg'); plt.ylabel('Sign Confidence')
    plt.legend(); plt.tight_layout(); plt.show()
    print('Nilai mendekati 0 = Generator berhasil membodohi Diskriminator')
else:
    print('Metrik Sign tidak tersedia di log ini (coba versi StyleGAN2-ADA yang lebih baru)')

In [ ]:
# VIS 8: Penalti Regularisasi R1 & Path Length
if len(df) and df.R1.abs().sum() > 0:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    sns.lineplot(x='Kimg', y='R1', data=df, ax=axes[0], color='purple', linewidth=2)
    axes[0].set_title('Vis 8a: R1 Gradient Penalty'); axes[0].set_xlabel('Kimg'); axes[0].set_ylabel('R1 Penalty')
    sns.lineplot(x='Kimg', y='PL', data=df, ax=axes[1], color='teal', linewidth=2)
    axes[1].set_title('Vis 8b: Path Length Regularization'); axes[1].set_xlabel('Kimg'); axes[1].set_ylabel('PL Penalty')
    plt.suptitle('Vis 8: Dinamika Penalti Regularisasi (Anti Mode Collapse)', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()
else:
    print('Metrik R1/PL tidak tersedia — lewat ke Vis 9')

In [ ]:
# VIS 9: Profil Penggunaan Memori Hardware (VRAM & RAM)
if len(df) and df.GPU.abs().sum() > 0:
    plt.figure(figsize=(14, 5))
    sns.lineplot(x='Kimg', y='GPU', data=df, label='VRAM GPU (GB)', color='crimson', linewidth=2)
    sns.lineplot(x='Kimg', y='CPU', data=df, label='RAM CPU (GB)', color='steelblue', linewidth=2)
    plt.title('Vis 9: Utilisasi Memori Perangkat Keras Kaggle T4 x2 Selama Training')
    plt.xlabel('Kimg'); plt.ylabel('Gigabytes (GB)')
    plt.legend(); plt.tight_layout(); plt.show()
else:
    print('Metrik memori tidak tersedia — lewat ke Vis 10')

In [ ]:
# VIS 10: Matriks Korelasi Pearson
if len(df) and len(df_fid):
    df_m = pd.merge_asof(df.sort_values('Kimg'), df_fid.sort_values('Kimg'), on='Kimg', direction='nearest')
    cols = [c for c in ['Loss_Fake','Loss_Real','Augment_P','Sign_Real','R1','PL','FID'] if df_m[c].abs().sum() > 0]
    plt.figure(figsize=(10, 8))
    sns.heatmap(df_m[cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1, linewidths=0.5)
    plt.title('Vis 10: Matriks Korelasi Kinerja Model (Pearson)')
    plt.tight_layout(); plt.show()
    print('Korelasi negatif Augment_P ↔ FID = bukti statistik khasiat ADA')
else:
    print('Butuh data log + FID untuk matriks korelasi')

### 🎁 D. Bonus: Generate Gambar HD & Video Morphing

In [ ]:
# BONUS 1: Generate 30 Gambar HD dari PKL Terbaik
!git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git vis_engine

# Patch custom_ops agar bersih
with open('vis_engine/torch_utils/custom_ops.py', 'r') as f: lines = f.readlines()
nl, i = [], 0
while i < len(lines):
    if 'torch.utils.cpp_extension.load(' in lines[i] and 'module =' not in lines[i]:
        nl.append(lines[i].replace('torch.utils.cpp_extension.load(', 'module = torch.utils.cpp_extension.load('))
        if i+1 < len(lines) and 'importlib.import_module' in lines[i+1]: i += 2; continue
    elif 'importlib.import_module(module_name)' in lines[i]: i += 1; continue
    else: nl.append(lines[i])
    i += 1
with open('vis_engine/torch_utils/custom_ops.py', 'w') as f: f.writelines(nl)

all_pkl = sorted(glob.glob('/kaggle/input/**/*.pkl', recursive=True))
snapshots = [f for f in all_pkl if 'network-snapshot-' in f]
if snapshots:
    BEST_PKL = snapshots[-1]
    os.makedirs('/kaggle/working/hasil_gambar', exist_ok=True)
    print(f'Generating dari: {BEST_PKL}')
    os.system(f'cd vis_engine && python -W ignore generate.py --outdir=/kaggle/working/hasil_gambar --trunc=0.7 --seeds=100-129 --network={BEST_PKL}')
    count = len(glob.glob('/kaggle/working/hasil_gambar/*.png'))
    print(f'✅ {count} gambar HD tersimpan di output!')

    # Tampilkan grid 4x4 INLINE di notebook
    imgs = sorted(glob.glob('/kaggle/working/hasil_gambar/*.png'))[:16]
    fig, axes = plt.subplots(4, 4, figsize=(16, 16))
    for ax, img_path in zip(axes.flat, imgs):
        ax.imshow(mpimg.imread(img_path)); ax.axis('off')
    plt.suptitle('Bonus 1: Grid 16 Gambar Serviks Generator (Truncation=0.7)', fontsize=16, fontweight='bold')
    plt.tight_layout(); plt.show()
else:
    print('Tambahkan dataset .pkl via Add Data untuk generate gambar baru')

In [ ]:
# BONUS 2: Video Animasi Morphing (W-Space Interpolation)
import sys
sys.path.insert(0, 'vis_engine')
import torch, dnnlib, legacy, imageio
from tqdm import tqdm

if 'BEST_PKL' in locals():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        with dnnlib.util.open_url(BEST_PKL) as f:
            G = legacy.load_network_pkl(f)['G_ema'].to(device)
    seeds = [1, 5, 10, 15]
    zs = torch.from_numpy(np.stack([np.random.RandomState(s).randn(G.z_dim) for s in seeds])).to(device)
    ws = G.mapping(zs, None, truncation_psi=0.7)
    os.makedirs('/kaggle/working/hasil_video', exist_ok=True)
    vp = '/kaggle/working/hasil_video/animasi_serviks_morph.mp4'
    vo = imageio.get_writer(vp, mode='I', fps=30, codec='libx264', bitrate='8M')
    frames_all = []
    for i in range(len(seeds)):
        w1, w2 = ws[i], ws[(i+1)%len(seeds)]
        for step in tqdm(range(45), desc=f'Transisi {i+1}/4'):
            alpha = step / 45.0
            w_i = torch.lerp(w1, w2, alpha).unsqueeze(0)
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                img = G.synthesis(w_i, noise_mode='const')
            img_np = (img.permute(0,2,3,1)*127.5+128).clamp(0,255).to(torch.uint8)[0].cpu().numpy()
            vo.append_data(img_np)
            frames_all.append(img_np)
    vo.close()
    print(f'✅ Video tersimpan: {vp}')
    # Tampilkan 8 keyframes INLINE
    idxs = np.linspace(0, len(frames_all)-1, 8, dtype=int)
    fig, axes = plt.subplots(1, 8, figsize=(24, 4))
    for ax, idx in zip(axes, idxs):
        ax.imshow(frames_all[idx]); ax.axis('off')
    plt.suptitle('Bonus 2: Keyframes Animasi Morphing Serviks (W-Space Interpolation)', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()
else:
    print('PKL tidak ditemukan')

In [ ]:
# PACKING ZIP
os.system('zip -r -q /kaggle/working/Visualisasi_Skripsi_Final.zip /kaggle/working/hasil_gambar /kaggle/working/hasil_video 2>/dev/null || true')
print('🎁 Visualisasi_Skripsi_Final.zip siap di panel Output kanan!')